# Logistic Regression NIST Cross-Dataset Validation

This notebook mirrors the NIST validation workflow used for the Neural Network model, but applies it to the Logistic Regression baseline model.

Validation goal:
1. Train the logistic regression model using the NFI labeled particle data.
2. Reformat the NIST data so its element columns match the training feature space.
3. Run the model on NIST non-shooter particles as a false-positive / specificity test.
4. Run the model on NIST shooter particles as a sensitivity contrast.
5. Compare predicted GSR probabilities by source and class.

**Important:** This notebook keeps the same leakage-control logic as the logistic regression notebook by using a group-aware split on `stub_id`. It also keeps `pb`, `sb`, and `ba` excluded to match the current baseline modeling approach.


In [8]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

## 1. Load NFI Training Data

This is the same labeled particle dataset used in the logistic regression baseline notebook.


In [9]:
# Adjust this path if needed based on where this notebook is saved
df = pd.read_parquet("../../data/processed/particle_labeled.parquet")

print(f"NFI labeled dataset shape: {df.shape}")
print("Label distribution:")
print(df["label"].value_counts(dropna=False))

NFI labeled dataset shape: (2801667, 95)
Label distribution:
label
Non_GSR      1216039
GSR          1078946
Ambiguous     506682
Name: count, dtype: int64


## 2. Prepare Logistic Regression Features and Target

This cell matches the baseline logistic regression setup: binary labels only, excluded metadata/leakage fields, excludes `pb`, `sb`, and `ba`, and keeps numeric features only.


In [10]:
# Copy the modeling dataframe
df_model = df.copy()

# Keep only binary labels
df_model = df_model[df_model["label"].isin(["GSR", "Non_GSR"])].copy()

# Create binary target
df_model["target"] = df_model["label"].map({"Non_GSR": 0, "GSR": 1})

# Columns to exclude from modeling
exclude_cols = [
    "stub_id",
    "particle_id",
    "label",
    "target",
    "relevance_class",
    "final_class",
    "class",
    "subclass",
    "particle_class",
    "pb",
    "sb",
    "ba",
]

# Drop excluded columns if they exist
X = df_model.drop(columns=[col for col in exclude_cols if col in df_model.columns])

# Keep only numeric columns
X = X.select_dtypes(include=["number"])

y = df_model["target"].astype(int)
groups = df_model["stub_id"]
training_features = X.columns.tolist()

print("Feature matrix shape:", X.shape)
print("Number of features:", X.shape[1])
print("Target distribution:")
print(y.value_counts(normalize=True))
print("First 20 training features:")
print(training_features[:20])

Feature matrix shape: (2294985, 86)
Number of features: 86
Target distribution:
target
0    0.529868
1    0.470132
Name: proportion, dtype: float64
First 20 training features:
['ac', 'ag', 'al', 'ar', 'as', 'at', 'au', 'b', 'bi', 'br', 'ca', 'cd', 'ce', 'cl', 'co', 'cr', 'cs', 'cu', 'dy', 'er']


## 3. Group-Aware Train/Test Split

The split prevents particles from the same `stub_id` from appearing in both the training and test sets.


In [11]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Unique training stubs:", groups_train.nunique())
print("Unique testing stubs:", groups_test.nunique())
print("Stub overlap:", len(set(groups_train).intersection(set(groups_test))))
print("Training target distribution:")
print(y_train.value_counts(normalize=True))
print("Testing target distribution:")
print(y_test.value_counts(normalize=True))

Training shape: (1851761, 86)
Testing shape: (443224, 86)
Unique training stubs: 3028
Unique testing stubs: 758
Stub overlap: 0
Training target distribution:
target
0    0.530719
1    0.469281
Name: proportion, dtype: float64
Testing target distribution:
target
0    0.526314
1    0.473686
Name: proportion, dtype: float64


## 4. Train Baseline Logistic Regression Model

This uses the baseline settings from your current logistic regression notebook.


In [12]:
baseline_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                solver="saga",
                C=0.5,
                class_weight="balanced",
                max_iter=5000,
                tol=0.001,
                random_state=42,
            ),
        ),
    ]
)

baseline_pipe.fit(X_train, y_train)

baseline_pred = baseline_pipe.predict(X_test)
baseline_proba = baseline_pipe.predict_proba(X_test)[:, 1]

baseline_results = {
    "Model": "Baseline Logistic Regression",
    "Regularization": "L2",
    "C": 0.5,
    "Class Weight": "balanced",
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred),
    "Recall": recall_score(y_test, baseline_pred),
    "F1": f1_score(y_test, baseline_pred),
    "ROC-AUC": roc_auc_score(y_test, baseline_proba),
    "PR-AUC": average_precision_score(y_test, baseline_proba),
}

pd.DataFrame([baseline_results])

,Model,Regularization,C,Class Weight,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Baseline Logistic Regression,L2,0.5,balanced,0.819913,0.788836,0.846387,0.816599,0.90048,0.879804


## 5. Confirm Held-Out Test Performance


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, baseline_proba, ax=ax)
ax.set_title("Baseline Logistic Regression ROC Curve on NFI Test Set")
ax.grid(True, alpha=0.3)
plt.savefig(
    "outputs/logistic_regression_nist_validation__baseline_logistic_regression_roc_curve_on_nfi_test_set.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
PrecisionRecallDisplay.from_predictions(y_test, baseline_proba, ax=ax)
ax.set_title("Baseline Logistic Regression Precision-Recall Curve on NFI Test Set")
ax.grid(True, alpha=0.3)
plt.show()

print("Classification report at default threshold = 0.50")
print(classification_report(y_test, baseline_pred, target_names=["Non_GSR", "GSR"]))

## 6. Define Thresholds for NIST Validation

The default threshold is 0.50. Additional thresholds are selected from the NFI held-out test set.


In [14]:
threshold_grid = np.linspace(0.01, 0.99, 99)
threshold_rows = []

for threshold in threshold_grid:
    threshold_pred = (baseline_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, threshold_pred).ravel()
    threshold_rows.append(
        {
            "Threshold": threshold,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
            "FNR": fn / (fn + tp) if (fn + tp) else np.nan,
            "Precision": precision_score(y_test, threshold_pred, zero_division=0),
            "Recall": recall_score(y_test, threshold_pred, zero_division=0),
            "F1": f1_score(y_test, threshold_pred, zero_division=0),
        }
    )

threshold_df = pd.DataFrame(threshold_rows)


def get_row_nearest_threshold(df, threshold):
    return df.loc[[(df["Threshold"] - threshold).abs().idxmin()]]


def get_best_f1_row(df):
    return df.loc[[df["F1"].idxmax()]]


def get_high_specificity_row(df, min_recall=0.70):
    candidates = df[df["Recall"] >= min_recall].copy()
    if candidates.empty:
        candidates = df.copy()
    idx = candidates.sort_values(
        ["FPR", "FN", "Threshold"], ascending=[True, True, False]
    ).index[0]
    return df.loc[[idx]]


def get_high_sensitivity_row(df, min_precision=0.10):
    candidates = df[df["Precision"] >= min_precision].copy()
    if candidates.empty:
        candidates = df.copy()
    idx = candidates.sort_values(
        ["FNR", "FP", "Threshold"], ascending=[True, True, True]
    ).index[0]
    return df.loc[[idx]]


default_row = get_row_nearest_threshold(threshold_df, 0.50).assign(
    Mode="Default Threshold"
)
best_f1_row = get_best_f1_row(threshold_df).assign(Mode="Balanced / Best F1")
high_specificity_row = get_high_specificity_row(threshold_df, min_recall=0.70).assign(
    Mode="High Specificity"
)
high_sensitivity_row = get_high_sensitivity_row(
    threshold_df, min_precision=0.10
).assign(Mode="High Sensitivity")

threshold_summary = pd.concat(
    [default_row, best_f1_row, high_specificity_row, high_sensitivity_row]
)
threshold_summary = threshold_summary[
    ["Mode", "Threshold", "FPR", "FNR", "Precision", "Recall", "F1", "FP", "FN"]
].reset_index(drop=True)

thresholds = {
    row["Mode"]: float(row["Threshold"]) for _, row in threshold_summary.iterrows()
}
threshold_summary

,Mode,Threshold,FPR,FNR,Precision,Recall,F1,FP,FN
0,Default Threshold,0.50,0.203914,0.153613,0.788836,0.846387,0.816599,47568,32251
1,Balanced / Best F1,0.60,0.112087,0.210904,0.863688,0.789096,0.824709,26147,44279
2,High Specificity,0.70,0.057400,0.299392,0.916564,0.700608,0.794167,13390,62857
3,High Sensitivity,0.01,0.928213,0.003796,0.491335,0.996204,0.658094,216529,797


## 7. Load NIST Data

This is the same NIST parquet file used in the neural network validation notebook.


In [16]:
# Adjust this path if needed based on where this notebook is saved
nist = pd.read_parquet(
    "../../data/processed/nist_concatenated_parquets/nist_neQuant_concatenated_data.parquet"
)

print(f"NIST shape: {nist.shape}")
print(f"Sample sources: {nist['sample_source'].nunique()}")
print("Sample source counts:")
print(nist["sample_source"].value_counts().to_string())

NIST shape: (135447, 103)
Sample sources: 30
Sample source counts:
sample_source
brakes_chevy_front_driver                      5000
brakes_chevy_front_passenger                   5000
brakes_chevy_rear_driver                       5000
brakes_chevy_rear_passenger                    5000
brakes_ford_a213_front_driver                  5000
brakes_ford_a213_front_passenger               5000
brakes_ford_a213_rear_driver                   5000
brakes_ford_a213_rear_passenger                5000
brakes_ford_b297_front_driver                  5000
brakes_ford_b297_front_passenger               5000
fireworks_romancandles_postcleanup             5000
fireworks_romancandles_posthandle_preignite    5000
fireworks_sparklers_burning                    5000
fireworks_sparklers_debris                     5000
fireworks_sparklers_posthandling_postburn      5000
fireworks_spinners_debris                      5000
fireworks_spinners_postcleanup                 5000
fireworks_spinners_posthandling_pre

## 8. Split NIST into Non-Shooter and Shooter Data


In [17]:
shooter_mask = nist["sample_source"].str.startswith("shooter")
nist_nonshooter = nist[~shooter_mask].copy()
nist_shooter = nist[shooter_mask].copy()

print(
    f"Non-shooter particles: {len(nist_nonshooter):,} ({nist_nonshooter['sample_source'].nunique()} sources)"
)
print(
    f"Shooter particles: {len(nist_shooter):,} ({nist_shooter['sample_source'].nunique()} sources)"
)
print("Non-shooter CLASS distribution:")
print(nist_nonshooter["CLASS"].value_counts().head(15).to_string())
print("Shooter CLASS distribution:")
print(nist_shooter["CLASS"].value_counts().head(15).to_string())

Non-shooter particles: 95,447 (22 sources)
Shooter particles: 40,000 (8 sources)
Non-shooter CLASS distribution:
CLASS
#Unclassified#        42718
Iron-bearing          18993
Ca-bearing             9842
Iron                   8934
Silicate               4867
Uranium-bearing        2498
Quartz-like            2303
Dolomite               1427
Zinc                   1135
Aluminosilicate         917
Other Silicate          540
Aluminum                302
Calcium silicate        282
Al-Mg alloy (5XXX)      130
Fe-Al alloy             118
Shooter CLASS distribution:
CLASS
Ca-bearing         16479
#Unclassified#      9958
Silicate            3830
Iron-bearing        2036
Iron                1893
Uranium-bearing     1183
Quartz-like          844
Char GSR-like        694
Dolomite             666
Aluminosilicate      495
Pb-Ba                264
Aluminum             206
Other Silicate       204
Ba-Sb                204
Zinc                 165


## 9. Preprocess NIST into Logistic Regression Feature Format

NIST uses uppercase element columns and `sum_with_oxygen`. This converts NIST values into lowercase percentage columns, then aligns them to the exact logistic regression training features. Training features unavailable in NIST are filled with 0.


In [18]:
nist_element_cols_upper = [
    "O",
    "MG",
    "AL",
    "SI",
    "P",
    "S",
    "CL",
    "K",
    "CA",
    "TI",
    "CR",
    "MN",
    "FE",
    "NI",
    "CU",
    "ZN",
    "RB",
    "SR",
    "ZR",
    "MO",
    "RH",
    "PD",
    "AG",
    "IN",
    "SN",
    "SB",
    "BA",
    "LA",
    "CE",
    "ND",
    "SM",
    "TB",
    "TM",
    "AU",
    "PB",
    "BI",
]
nist_element_cols_upper = [
    col for col in nist_element_cols_upper if col in nist.columns
]


def preprocess_nist_for_logistic(df, training_features):
    result = pd.DataFrame(index=df.index)
    if "sum_with_oxygen" not in df.columns:
        raise ValueError("Expected NIST column sum_with_oxygen was not found.")

    sum_with_o = df["sum_with_oxygen"].replace(0, np.nan)
    for nist_col in nist_element_cols_upper:
        lower_name = nist_col.lower()
        result[lower_name] = (df[nist_col] / sum_with_o) * 100.0

    result["sample_source"] = df["sample_source"].values
    result["CLASS"] = df["CLASS"].values

    X_nist = result.reindex(columns=training_features, fill_value=0)
    X_nist = X_nist.apply(pd.to_numeric, errors="coerce").fillna(0)
    return result, X_nist


nist_nonshooter_proc, X_nist_nonshooter = preprocess_nist_for_logistic(
    nist_nonshooter, training_features
)
nist_shooter_proc, X_nist_shooter = preprocess_nist_for_logistic(
    nist_shooter, training_features
)

missing_training_features = sorted(
    set(training_features) - set(nist_nonshooter_proc.columns)
)
extra_nist_features = sorted(
    set(nist_nonshooter_proc.columns)
    - set(training_features)
    - {"sample_source", "CLASS"}
)

print("NIST non-shooter model matrix:", X_nist_nonshooter.shape)
print("NIST shooter model matrix:", X_nist_shooter.shape)
print(
    f"Number of training features unavailable in NIST and filled with 0: {len(missing_training_features)}"
)
print(missing_training_features)
print(f"NIST element columns not used by logistic model: {len(extra_nist_features)}")
print(extra_nist_features)

NIST non-shooter model matrix: (95447, 86)
NIST shooter model matrix: (40000, 86)
Number of training features unavailable in NIST and filled with 0: 53
['ac', 'ar', 'as', 'at', 'b', 'br', 'cd', 'co', 'cs', 'dy', 'er', 'eu', 'f', 'fr', 'ga', 'gd', 'ge', 'hf', 'hg', 'ho', 'i', 'ir', 'kr', 'lu', 'n', 'na', 'nb', 'ne', 'np', 'os', 'pa', 'pm', 'po', 'pr', 'pt', 'pu', 'ra', 're', 'rn', 'ru', 'sc', 'se', 'ta', 'tc', 'te', 'th', 'tl', 'u', 'v', 'w', 'xe', 'y', 'yb']
NIST element columns not used by logistic model: 3
['ba', 'pb', 'sb']


## 10. Run Logistic Regression on NIST Non-Shooter Data

Since these are non-shooter particles, predicted GSR classifications are treated as false positives.


In [19]:
nonshooter_probs = baseline_pipe.predict_proba(X_nist_nonshooter)[:, 1]
print(f"Non-shooter predictions computed: {len(nonshooter_probs):,} particles")

for name, thresh in thresholds.items():
    preds = (nonshooter_probs >= thresh).astype(int)
    n_fp = preds.sum()
    fpr = n_fp / len(preds)
    print(
        f"Threshold={thresh:.4f} ({name}): {n_fp:,} false positives out of {len(preds):,} ({fpr:.4%})"
    )

Non-shooter predictions computed: 95,447 particles
Threshold=0.5000 (Default Threshold): 5,571 false positives out of 95,447 (5.8367%)
Threshold=0.6000 (Balanced / Best F1): 4,963 false positives out of 95,447 (5.1997%)
Threshold=0.7000 (High Specificity): 4,446 false positives out of 95,447 (4.6581%)
Threshold=0.0100 (High Sensitivity): 44,950 false positives out of 95,447 (47.0942%)


### Non-Shooter False Positives by Source


In [20]:
sources = nist_nonshooter_proc["sample_source"].values
nonshooter_preds = (nonshooter_probs >= thresholds["Default Threshold"]).astype(int)

source_results = []
for src in sorted(np.unique(sources)):
    mask = sources == src
    n = mask.sum()
    n_fp = nonshooter_preds[mask].sum()
    mean_p = nonshooter_probs[mask].mean()
    source_type = (
        "brakes" if "brakes" in src else "fireworks" if "fireworks" in src else "other"
    )
    source_results.append(
        {
            "Source": src,
            "Type": source_type,
            "N": n,
            "FP": n_fp,
            "FPR": n_fp / n,
            "Mean P(GSR)": mean_p,
        }
    )

source_results_df = pd.DataFrame(source_results)
display(source_results_df)

for stype in ["brakes", "fireworks", "other"]:
    mask = source_results_df["Type"].eq(stype)
    if mask.any():
        n = source_results_df.loc[mask, "N"].sum()
        n_fp = source_results_df.loc[mask, "FP"].sum()
        print(f"{stype.upper()} total: {n_fp:,} FP out of {n:,} ({n_fp / n:.4%})")

,Source,Type,N,FP,FPR,Mean P(GSR)
0,brakes_chevy_front_driver,brakes,5000,11,0.002200,0.033487
1,brakes_chevy_front_passenger,brakes,5000,9,0.001800,0.032713
2,brakes_chevy_rear_driver,brakes,5000,120,0.024000,0.072545
3,brakes_chevy_rear_passenger,brakes,5000,58,0.011600,0.063811
4,brakes_ford_a213_front_driver,brakes,5000,63,0.012600,0.042910
5,brakes_ford_a213_front_passenger,brakes,5000,40,0.008000,0.030237
6,brakes_ford_a213_rear_driver,brakes,5000,139,0.027800,0.077027
7,brakes_ford_a213_rear_passenger,brakes,5000,76,0.015200,0.055517
8,brakes_ford_b297_front_driver,brakes,5000,32,0.006400,0.046957
9,brakes_ford_b297_front_passenger,brakes,5000,66,0.013200,0.055343


BRAKES total: 614 FP out of 50,200 (1.2231%)
FIREWORKS total: 4,957 FP out of 45,247 (10.9554%)


### Non-Shooter False Positives by NIST Class


In [21]:
if nonshooter_preds.sum() > 0:
    print("False-positive by NIST CLASS breakdown at default threshold = 0.50")
    fp_mask = nonshooter_preds == 1
    fp_classes = nist_nonshooter_proc["CLASS"].values[fp_mask]
    fp_class_counts = pd.Series(fp_classes).value_counts()
    display(fp_class_counts.rename("False Positives").to_frame())
else:
    print("No false positives in non-shooter data at default threshold = 0.50.")

False-positive by NIST CLASS breakdown at default threshold = 0.50


,False Positives
#Unclassified#,5373
Uranium-bearing,96
Iron-bearing,68
Cu-rich,17
Ca-bearing,12
Iron,3
Silicate,2


## 11. Run Logistic Regression on NIST Shooter Data

This provides a sensitivity contrast. A model that generalizes well should assign higher GSR probabilities to shooter particles than to non-shooter particles.


In [22]:
shooter_probs = baseline_pipe.predict_proba(X_nist_shooter)[:, 1]
print(f"Shooter predictions computed: {len(shooter_probs):,} particles")

for name, thresh in thresholds.items():
    preds = (shooter_probs >= thresh).astype(int)
    n_gsr = preds.sum()
    gsr_rate = n_gsr / len(preds)
    print(
        f"Threshold={thresh:.4f} ({name}): {n_gsr:,} classified as GSR out of {len(preds):,} ({gsr_rate:.2%})"
    )

Shooter predictions computed: 40,000 particles
Threshold=0.5000 (Default Threshold): 1,627 classified as GSR out of 40,000 (4.07%)
Threshold=0.6000 (Balanced / Best F1): 1,388 classified as GSR out of 40,000 (3.47%)
Threshold=0.7000 (High Specificity): 1,168 classified as GSR out of 40,000 (2.92%)
Threshold=0.0100 (High Sensitivity): 8,134 classified as GSR out of 40,000 (20.34%)


### Shooter GSR Rate by Source


In [23]:
shooter_sources = nist_shooter_proc["sample_source"].values
shooter_preds = (shooter_probs >= thresholds["Default Threshold"]).astype(int)

shooter_results = []
for src in sorted(np.unique(shooter_sources)):
    mask = shooter_sources == src
    n = mask.sum()
    n_gsr = shooter_preds[mask].sum()
    mean_p = shooter_probs[mask].mean()
    shooter_results.append(
        {
            "Source": src,
            "N": n,
            "GSR": n_gsr,
            "GSR Rate": n_gsr / n,
            "Mean P(GSR)": mean_p,
        }
    )

shooter_results_df = pd.DataFrame(shooter_results)
display(shooter_results_df)

,Source,N,GSR,GSR Rate,Mean P(GSR)
0,shooter_1,5000,287,0.0574,0.068372
1,shooter_2,5000,346,0.0692,0.074454
2,shooter_3L,5000,281,0.0562,0.068484
3,shooter_3R,5000,224,0.0448,0.056941
4,shooter_4L,5000,45,0.0090,0.014894
5,shooter_4R,5000,47,0.0094,0.014171
6,shooter_5L,5000,146,0.0292,0.033115
7,shooter_5R,5000,251,0.0502,0.059608


### Shooter GSR Rate by NIST Class


In [24]:
shooter_classes = nist_shooter_proc["CLASS"].values
class_results = []

for cls in sorted(np.unique(shooter_classes)):
    mask = shooter_classes == cls
    n = mask.sum()
    if n < 10:
        continue
    n_gsr = shooter_preds[mask].sum()
    mean_p = shooter_probs[mask].mean()
    class_results.append(
        {
            "CLASS": cls,
            "N": n,
            "GSR": n_gsr,
            "GSR Rate": n_gsr / n,
            "Mean P(GSR)": mean_p,
        }
    )

class_results_df = pd.DataFrame(class_results)
display(class_results_df)

,CLASS,N,GSR,GSR Rate,Mean P(GSR)
0,#Unclassified#,9958,949,0.095300,0.106749
1,Aluminosilicate,495,0,0.000000,0.004922
2,Aluminum,206,0,0.000000,0.000815
3,Ba-Sb,204,84,0.411765,0.405735
4,Brass,81,1,0.012346,0.072382
5,Ca-bearing,16479,12,0.000728,0.002014
6,Calcium silicate,68,0,0.000000,0.000572
7,Char GSR-like,694,262,0.377522,0.399575
8,Cr-bearing,26,0,0.000000,0.023164
9,Cu-rich,136,61,0.448529,0.466595


## 12. Probability Distribution Comparison


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(nonshooter_probs, bins=100, alpha=0.7, label="Non-shooter", density=True)
axes[0].hist(shooter_probs, bins=100, alpha=0.7, label="Shooter", density=True)
axes[0].set_xlabel("P(GSR)")
axes[0].set_ylabel("Density")
axes[0].set_title("P(GSR) Distribution: Non-Shooter vs Shooter")
axes[0].legend()
axes[0].set_yscale("log")
axes[0].grid(True, alpha=0.3)

axes[1].hist(nonshooter_probs, bins=100, alpha=0.7, edgecolor="black")
axes[1].set_xlabel("P(GSR)")
axes[1].set_ylabel("Count")
axes[1].set_title("Non-Shooter P(GSR) Distribution")
axes[1].axvline(
    x=thresholds["Default Threshold"],
    linestyle="--",
    alpha=0.7,
    label="Default threshold",
)
axes[1].legend()
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)

axes[2].hist(shooter_probs, bins=100, alpha=0.7, edgecolor="black")
axes[2].set_xlabel("P(GSR)")
axes[2].set_ylabel("Count")
axes[2].set_title("Shooter P(GSR) Distribution")
axes[2].axvline(
    x=thresholds["Default Threshold"],
    linestyle="--",
    alpha=0.7,
    label="Default threshold",
)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    "outputs/logistic_regression_nist_validation__p_gsr_distribution_non_shooter_vs_shooter.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 13. Summary Statistics


In [26]:
print("Summary stats")
print(f"Non-shooter:")
print(f"N: {len(nonshooter_probs):,}")
print(f"Mean P(GSR): {nonshooter_probs.mean():.6f}")
print(f"Median P(GSR): {np.median(nonshooter_probs):.6f}")
print(f"P(GSR) < 0.1: {(nonshooter_probs < 0.1).mean():.2%}")
print(f"P(GSR) > 0.5: {(nonshooter_probs > 0.5).mean():.2%}")

print(f"Shooter:")
print(f"N: {len(shooter_probs):,}")
print(f"Mean P(GSR): {shooter_probs.mean():.6f}")
print(f"Median P(GSR): {np.median(shooter_probs):.6f}")
print(f"P(GSR) < 0.1: {(shooter_probs < 0.1).mean():.2%}")
print(f"P(GSR) > 0.5: {(shooter_probs > 0.5).mean():.2%}")

nonshooter_gsr_rate = (nonshooter_probs >= thresholds["Default Threshold"]).mean()
shooter_gsr_rate = (shooter_probs >= thresholds["Default Threshold"]).mean()

print(f"Contrast at default threshold = {thresholds['Default Threshold']:.2f}:")
print(f"Non-shooter GSR rate: {nonshooter_gsr_rate:.4%}")
print(f"Shooter GSR rate: {shooter_gsr_rate:.2%}")
print(f"Ratio: {shooter_gsr_rate / max(nonshooter_gsr_rate, 1e-10):.1f}x")

Summary stats
Non-shooter:
N: 95,447
Mean P(GSR): 0.080843
Median P(GSR): 0.008071
P(GSR) < 0.1: 85.86%
P(GSR) > 0.5: 5.84%
Shooter:
N: 40,000
Mean P(GSR): 0.048755
Median P(GSR): 0.000739
P(GSR) < 0.1: 91.42%
P(GSR) > 0.5: 4.07%
Contrast at default threshold = 0.50:
Non-shooter GSR rate: 5.8367%
Shooter GSR rate: 4.07%
Ratio: 0.7x


## 14. Interpretation Summary

The logistic regression model demonstrates strong performance on the original NFI held-out test set (ROC-AUC ≈ 0.90, PR-AUC ≈ 0.88), confirming that a linear model can effectively distinguish GSR from Non-GSR particles when clear multi-element signatures are present. However, the cross-dataset NIST validation provides a more realistic assessment of generalization under challenging, real-world conditions.

On the NIST non-shooter dataset, the model produces a measurable false positive rate. At the default threshold (0.50), approximately 5,447 out of 95,447 non-shooter particles (~5.8%) are incorrectly classified as GSR . These false positives are not uniformly distributed; instead, they are concentrated in specific sources—most notably fireworks (~10.95%) and, to a lesser extent, brake-related particles (~1.22%) . This indicates that certain environmental or mechanical particle types exhibit chemical compositions that partially overlap with GSR signatures, creating predictable areas of model confusion.

On the NIST shooter dataset, the model correctly assigns higher GSR probabilities to true GSR particles, but the overall GSR classification rate remains modest (≈4.1% at the default threshold) . This reflects a conservative classification boundary, which limits false positives but also reduces sensitivity. Threshold tuning demonstrates expected trade-offs: lower thresholds increase sensitivity but sharply raise false positives, while higher thresholds improve specificity at the cost of missed detections.

The probability distribution comparison further reinforces these findings. Non-shooter particles cluster heavily at low predicted probabilities, while shooter particles show a broader distribution with a heavier upper tail. However, there remains overlap between the two groups, indicating that perfect separation is not achievable with the current feature space.

Importantly, the exclusion of dominant elements (Pb, Sb, Ba) reduces artificial separability and forces the model to rely on more subtle multi-element patterns. While this lowers peak performance on the NFI dataset, it provides a more realistic evaluation of how the model behaves when clear GSR signatures are absent or partially degraded.

Overall, the NIST validation confirms that the logistic regression model generalizes reasonably well but is sensitive to specific non-GSR particle types that chemically resemble GSR. The results highlight that:

- Model performance is driven more by feature representation than algorithm complexity
- False positives are source-dependent, not random
- Threshold selection is critical and must be aligned with operational priorities (sensitivity vs. specificity)
- The current feature space captures meaningful signal but does not fully eliminate overlap between GSR and certain non-GSR particles

These findings suggest that future improvements should focus on enhanced feature engineering (e.g., interaction terms, compositional transformations) and potentially ensemble or nonlinear models to better capture subtle distinctions between chemically similar particle classes.